# GH-ANFIS_E403 GRS Rule Viewer

- `Primary`는 GH-ANFIS의 `base` rule set입니다.
- `Complementary`는 GH-ANFIS의 `residual` rule set입니다.
- 각 규칙을 `IF ... THEN ...` 형태로 출력하고, 표(`DataFrame`)와 파일(`csv`, `txt`)로도 저장합니다.
- E403에서 저장된 `gh_anfis.pt` 체크포인트의 두 포맷을 모두 지원합니다.
  - `model_config` + `model_state_dict`
  - `n_features/n_outputs` + `params` + `state_dict`

`THEN` 절의 클래스는 학습 시 인코딩된 클래스 인덱스 기준입니다. 이진 분류에서는 `class_1 probability`를 함께 표시합니다.


In [1]:
from pathlib import Path
import json
import os
import sys
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()

def _looks_like_project_root(path: Path) -> bool:
    markers = ['model.py', 'data.py', 'learning.py', 'gh_config.py']
    return all((path / marker).exists() for marker in markers)

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E403',
        PROJECT_ROOT / 'GH-ANFIS_E403',
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS
from gh_config import normalize_gh_params
from data import (
    load_bcwd_data,
    load_gisette_data,
    load_spambase_data,
    load_vowel_data,
    coerce_numeric_frame,
    drop_nan_targets,
)

PROJECT_ROOT


PosixPath('/home/harp3133t/Research/03_Research/GH-ANFIS_E403')

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CV_WEIGHT_ROOT = PROJECT_ROOT / 'hyper_parameter' / 'cv_weights'

AVAILABLE_GH_CHECKPOINTS = sorted(CV_WEIGHT_ROOT.glob('*/fold_*/gh_anfis.pt'))
AVAILABLE_DATASET_KEYS = sorted({path.parent.parent.name for path in AVAILABLE_GH_CHECKPOINTS})

DATASET_KEY = AVAILABLE_DATASET_KEYS[0] if AVAILABLE_DATASET_KEYS else 'Breast_Cancer_Wisconsin__Original___no_mi'
FOLD_IDX = 1
CHECKPOINT_PATH = None  # 예: PROJECT_ROOT / 'hyper_parameter' / 'cv_weights' / DATASET_KEY / 'fold_01' / 'gh_anfis.pt'

TOP_TERMS_PER_RULE = 8
COMPUTE_MEAN_RULE_ACTIVATION = True
EXPORT_DIR = PROJECT_ROOT / 'output' / 'grs_rule_if_then'

print('DEVICE =', DEVICE)
print('CV_WEIGHT_ROOT =', CV_WEIGHT_ROOT)
print('Available dataset keys:')
for key in AVAILABLE_DATASET_KEYS:
    print('  -', key)


DEVICE = cuda
CV_WEIGHT_ROOT = /home/harp3133t/Research/03_Research/GH-ANFIS_E403/hyper_parameter/cv_weights
Available dataset keys:
  - Breast_Cancer_Wisconsin__Original___gh_only_5fold
  - Breast_Cancer_Wisconsin__Original___no_mi
  - Gisette__no_mi
  - Spambase__no_mi
  - Vowel__no_mi


In [3]:
def sigmoid_np(x: np.ndarray | float) -> np.ndarray | float:
    x_clip = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x_clip))


def softmax_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    denom = np.sum(e)
    if denom <= 0:
        return np.full_like(e, 1.0 / max(len(e), 1), dtype=np.float64)
    return e / denom


def list_fold_checkpoints(dataset_key: str) -> list[Path]:
    return sorted((CV_WEIGHT_ROOT / str(dataset_key)).glob('fold_*/gh_anfis.pt'))


def resolve_checkpoint_path(dataset_key: str, fold_idx: int, checkpoint_path: str | Path | None = None) -> Path:
    if checkpoint_path is not None:
        path = Path(checkpoint_path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f'Checkpoint not found: {path}')
        return path

    path = CV_WEIGHT_ROOT / str(dataset_key) / f'fold_{int(fold_idx):02d}' / 'gh_anfis.pt'
    if not path.exists():
        available = list_fold_checkpoints(dataset_key)
        raise FileNotFoundError(
            f'Checkpoint not found: {path}\nAvailable folds for {dataset_key}: {[p.parent.name for p in available]}'
        )
    return path.resolve()


def _infer_model_config(payload: dict[str, Any], params: dict[str, Any]) -> dict[str, Any]:
    cfg = payload.get('model_config')
    if cfg is not None:
        return dict(cfg)

    n_features = payload.get('n_features')
    n_outputs = payload.get('n_outputs')
    if n_features is None or n_outputs is None:
        raise KeyError('Checkpoint must contain either model_config or n_features/n_outputs.')

    return {
        'n_features': int(n_features),
        'n_outputs': int(n_outputs),
        'base_rules': int(params['base_rules']),
        'residual_rules': int(params['residual_rules']),
        'mf_per_feature': int(params['mf_per_feature']),
    }


def load_gh_checkpoint(checkpoint_path: Path, device: torch.device):
    payload = torch.load(checkpoint_path, map_location=device)
    if not isinstance(payload, dict):
        raise TypeError(f'Unsupported checkpoint payload type: {type(payload)}')

    raw_params = payload.get('hparams') or payload.get('params') or {}
    params = normalize_gh_params(raw_params)
    cfg = _infer_model_config(payload, params)

    state_dict = payload.get('model_state_dict') or payload.get('state_dict')
    if state_dict is None:
        raise KeyError('Checkpoint is missing model_state_dict/state_dict.')

    model = GH_ANFIS(
        n_features=int(cfg['n_features']),
        n_outputs=int(cfg['n_outputs']),
        base_rules=int(cfg['base_rules']),
        residual_rules=int(cfg['residual_rules']),
        mf_per_feature=int(cfg['mf_per_feature']),
        device=device,
        residual_gate_mode=str(params.get('residual_gate_mode', 'complement')),
        rule_init_mode=str(params.get('rule_init_mode', 'balanced')),
        rule_seed=int(params.get('rule_seed', 0)),
        firing_mode=str(params.get('firing_mode', 'htsk')),
        use_input_norm=bool(params.get('use_input_norm', False)),
        enable_residual_branch=bool(params.get('enable_residual_branch', True)),
    ).to(device)
    model.load_state_dict(state_dict)

    if params.get('base_mask_threshold') is not None:
        model.base_mask_threshold = float(params['base_mask_threshold'])
    if params.get('residual_mask_threshold') is not None:
        model.residual_mask_threshold = float(params['residual_mask_threshold'])

    model.base_mask_frozen = bool(params.get('base_hard_epochs', 0) or params.get('random_role_assignment', False))
    model.residual_mask_frozen = bool(params.get('residual_hard_epochs', 0) or params.get('random_role_assignment', False))

    model.eval()
    model.set_phase('residual_complement')
    model.set_mode('full')

    feature_names = payload.get('feature_names')
    if feature_names is None:
        feature_names = [f'f{i}' for i in range(model.n_features)]
    else:
        feature_names = list(feature_names)

    return model, feature_names, params, payload


def load_snapshot_bundle(dataset_key: str):
    if str(dataset_key).startswith('Breast_Cancer_Wisconsin'):
        X_df, y, feature_names = load_bcwd_data()
    elif str(dataset_key).startswith('Vowel'):
        X_df, y, feature_names = load_vowel_data()
    elif str(dataset_key).startswith('Spambase'):
        X_df, y, feature_names = load_spambase_data()
    elif str(dataset_key).startswith('Gisette'):
        X_df, y, feature_names = load_gisette_data()
    else:
        raise ValueError(f'Unsupported dataset key: {dataset_key}')

    X_df = coerce_numeric_frame(X_df)
    X_df, y = drop_nan_targets(X_df, y)
    if not hasattr(X_df, 'columns'):
        X_df = pd.DataFrame(X_df, columns=feature_names)
    return X_df.copy(), np.asarray(y), list(X_df.columns)


def align_snapshot_to_checkpoint(X_df: pd.DataFrame, checkpoint_feature_names: list[Any]) -> pd.DataFrame:
    missing = [col for col in checkpoint_feature_names if col not in X_df.columns]
    if missing:
        raise KeyError(f'Missing columns in snapshot data: {missing[:10]}')
    return X_df.loc[:, checkpoint_feature_names].copy()


def apply_saved_scaler(X_df: pd.DataFrame, payload: dict[str, Any], device: torch.device) -> torch.Tensor:
    X_arr = np.asarray(X_df.values, dtype=np.float32)
    mean = payload.get('scaler_mean')
    scale = payload.get('scaler_scale')
    if mean is not None and scale is not None:
        mean_arr = np.asarray(mean, dtype=np.float32)
        scale_arr = np.asarray(scale, dtype=np.float32)
        safe_scale = np.where(scale_arr == 0, 1.0, scale_arr)
        X_arr = (X_arr - mean_arr) / safe_scale
    return torch.tensor(X_arr, dtype=torch.float32, device=device)


@torch.no_grad()
def collect_rule_activation_stats(model: GH_ANFIS, x_tensor: torch.Tensor, batch_size: int = 512):
    model.eval()
    model.set_phase('residual_complement')
    model.set_mode('full')

    base_weights = []
    residual_weights = []
    n = int(x_tensor.shape[0])

    for start in range(0, n, batch_size):
        xb = x_tensor[start:start + batch_size]
        _, acts = model(xb, return_activations=True, use_soft_eval=False)
        bw = acts.get('base_rule_weights')
        rw = acts.get('residual_rule_weights')
        if bw is not None:
            base_weights.append(bw.detach().cpu().numpy())
        if rw is not None:
            residual_weights.append(rw.detach().cpu().numpy())

    base_mean = np.concatenate(base_weights, axis=0).mean(axis=0) if base_weights else None
    residual_mean = np.concatenate(residual_weights, axis=0).mean(axis=0) if residual_weights else None
    return base_mean, residual_mean


In [4]:
def gh_branch_masks(model: GH_ANFIS) -> dict[str, np.ndarray]:
    base_soft = torch.sigmoid(model.base_mask_logits).detach().cpu().numpy()
    base_hard = (base_soft >= float(model.base_mask_threshold)).astype(float)

    residual_soft = torch.sigmoid(model.residual_mask_logits).detach().cpu().numpy()
    residual_hard = (residual_soft >= float(model.residual_mask_threshold)).astype(float)

    if bool(model.residual_use_complement):
        residual_effective_hard = residual_hard * (1.0 - base_hard)
    else:
        residual_effective_hard = residual_hard.copy()

    return {
        'base_soft': base_soft,
        'base_hard': base_hard,
        'residual_soft': residual_soft,
        'residual_hard': residual_hard,
        'residual_effective_hard': residual_effective_hard,
    }


def _module_pack(model: GH_ANFIS, module: str) -> dict[str, Any]:
    masks = gh_branch_masks(model)
    if module == 'base':
        return {
            'module': 'base',
            'module_label': 'Primary',
            'display_prefix': 'P',
            'internal_prefix': 'B',
            'centers': model.s_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.s_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.base_consequents,
            'gate_hard': masks['base_hard'],
            'n_rules': int(model.base_rules),
        }
    if module == 'residual':
        return {
            'module': 'residual',
            'module_label': 'Complementary',
            'display_prefix': 'C',
            'internal_prefix': 'R',
            'centers': model.p_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.p_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.residual_consequents,
            'gate_hard': masks['residual_effective_hard'],
            'n_rules': int(model.residual_rules),
        }
    raise ValueError("module must be 'base' or 'residual'")


def infer_term_label(centers_1d: np.ndarray, mf_idx: int) -> tuple[str, int]:
    low_idx = int(np.argmin(centers_1d))
    high_idx = int(np.argmax(centers_1d))
    if mf_idx == low_idx:
        return 'low', -1
    if mf_idx == high_idx:
        return 'high', 1
    return f'mid(mf{int(mf_idx) + 1})', 0


def compute_rule_result(mp: dict[str, Any], rule_idx: int, x_proto: np.ndarray, n_outputs: int) -> tuple[np.ndarray, np.ndarray, int]:
    x_aug = np.concatenate([x_proto.astype(np.float32), np.array([1.0], dtype=np.float32)], axis=0)

    logits = []
    for out_idx in range(int(n_outputs)):
        layer = mp['consequents'][out_idx * mp['n_rules'] + rule_idx]
        weight = layer.weight.detach().cpu().numpy().reshape(-1)
        bias = 0.0
        if layer.bias is not None:
            bias = float(layer.bias.detach().cpu().numpy().reshape(-1)[0])
        logits.append(float(np.dot(weight, x_aug) + bias))

    logits = np.asarray(logits, dtype=np.float64)
    if int(n_outputs) == 1:
        p1 = float(sigmoid_np(logits[0]))
        probs = np.asarray([1.0 - p1, p1], dtype=np.float64)
        pred_class = int(p1 >= 0.5)
    else:
        probs = softmax_np(logits)
        pred_class = int(np.argmax(probs))
    return logits, probs, pred_class


def format_consequent_text(probs: np.ndarray, pred_class: int, logits: np.ndarray, n_outputs: int) -> str:
    if int(n_outputs) == 1:
        return (
            f'class_1 probability = {probs[1]:.4f}, '
            f'predicted_class = class_{pred_class}, '
            f'logit = {logits[0]:.4f}'
        )

    prob_text = ', '.join([f'class_{idx}={prob:.4f}' for idx, prob in enumerate(probs)])
    logit_text = ', '.join([f'class_{idx}={logit:.4f}' for idx, logit in enumerate(logits)])
    return f'predicted_class = class_{pred_class}, probs = [{prob_text}], logits = [{logit_text}]'


def module_rule_tables(
    model: GH_ANFIS,
    feature_names: list[Any],
    module: str,
    mean_rule_weights: np.ndarray | None = None,
    top_terms: int = 8,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    mp = _module_pack(model, module)

    selector_logits = np.asarray(mp['selector_logits'], dtype=np.float64)
    selector_logits = selector_logits - selector_logits.max(axis=-1, keepdims=True)
    selector_probs = np.exp(selector_logits)
    selector_probs = selector_probs / selector_probs.sum(axis=-1, keepdims=True)

    centers = np.asarray(mp['centers'], dtype=np.float64)
    gate_hard = np.asarray(mp['gate_hard'], dtype=np.float64)

    rule_rows = []
    term_rows = []

    for r in range(mp['n_rules']):
        best_mf = selector_probs[r].argmax(axis=1)
        best_prob = selector_probs[r].max(axis=1)

        x_proto = np.zeros(len(feature_names), dtype=np.float32)
        for d in range(len(feature_names)):
            x_proto[d] = float(centers[d, int(best_mf[d])] * gate_hard[d])

        logits, probs, pred_class = compute_rule_result(mp, r, x_proto, model.n_outputs)

        active_idx = np.where(gate_hard > 0.5)[0].tolist()
        if active_idx:
            active_idx = sorted(active_idx, key=lambda d: float(best_prob[d]), reverse=True)
        else:
            active_idx = np.argsort(best_prob)[::-1].tolist()
        active_idx = active_idx[: max(1, int(top_terms))]

        per_rule_terms = []
        for d in active_idx:
            term_label, term_sign = infer_term_label(centers[d], int(best_mf[d]))
            term_text = f'{feature_names[d]} is {term_label}'
            row = {
                'module': module,
                'module_label': mp['module_label'],
                'rule_idx': int(r),
                'display_rule_id': f"{mp['display_prefix']}{r + 1}",
                'internal_rule_id': f"{mp['internal_prefix']}{r + 1}",
                'feature': feature_names[d],
                'selector_prob': float(best_prob[d]),
                'selected_mf_idx': int(best_mf[d]),
                'selected_center': float(centers[d, int(best_mf[d])]),
                'term_label': term_label,
                'term_polarity_sign': int(term_sign),
                'term_text': term_text,
                'predicted_class': int(pred_class),
                'class_1_probability': float(probs[1]) if len(probs) > 1 else np.nan,
                'consequent_text': format_consequent_text(probs, pred_class, logits, model.n_outputs),
                'mean_rule_activation': float(mean_rule_weights[r]) if mean_rule_weights is not None and r < len(mean_rule_weights) else np.nan,
            }
            per_rule_terms.append(row)
            term_rows.append(row)

        antecedent_text = ' AND '.join([row['term_text'] for row in per_rule_terms]) if per_rule_terms else '(no active terms)'
        consequent_text = format_consequent_text(probs, pred_class, logits, model.n_outputs)
        if_then_text = f'IF {antecedent_text} THEN {consequent_text}'

        rule_rows.append(
            {
                'module': module,
                'module_label': mp['module_label'],
                'rule_idx': int(r),
                'display_rule_id': f"{mp['display_prefix']}{r + 1}",
                'internal_rule_id': f"{mp['internal_prefix']}{r + 1}",
                'active_feature_count': int(np.sum(gate_hard > 0.5)),
                'antecedent_text': antecedent_text,
                'consequent_text': consequent_text,
                'if_then_text': if_then_text,
                'predicted_class': int(pred_class),
                'class_1_probability': float(probs[1]) if len(probs) > 1 else np.nan,
                'class_probs': json.dumps({f'class_{idx}': float(prob) for idx, prob in enumerate(probs)}, ensure_ascii=False),
                'rule_logits': json.dumps({f'class_{idx}': float(logit) for idx, logit in enumerate(np.atleast_1d(logits))}, ensure_ascii=False),
                'mean_rule_activation': float(mean_rule_weights[r]) if mean_rule_weights is not None and r < len(mean_rule_weights) else np.nan,
            }
        )

    return pd.DataFrame(rule_rows), pd.DataFrame(term_rows)


def print_if_then_rules(rule_df: pd.DataFrame, title: str):
    print(f'[{title}]')
    if rule_df.empty:
        print('  (no rules)')
        return
    for _, row in rule_df.sort_values('rule_idx').iterrows():
        print(f"- {row['display_rule_id']} ({row['internal_rule_id']}): {row['if_then_text']}")


def export_rule_outputs(
    export_dir: Path,
    stem: str,
    primary_rules_df: pd.DataFrame,
    primary_terms_df: pd.DataFrame,
    complementary_rules_df: pd.DataFrame,
    complementary_terms_df: pd.DataFrame,
    checkpoint_info: dict[str, Any],
) -> dict[str, Path]:
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    primary_rules_csv = export_dir / f'{stem}_primary_rules.csv'
    primary_terms_csv = export_dir / f'{stem}_primary_terms.csv'
    complementary_rules_csv = export_dir / f'{stem}_complementary_rules.csv'
    complementary_terms_csv = export_dir / f'{stem}_complementary_terms.csv'
    summary_json = export_dir / f'{stem}_summary.json'
    if_then_txt = export_dir / f'{stem}_if_then_rules.txt'

    primary_rules_df.to_csv(primary_rules_csv, index=False)
    primary_terms_df.to_csv(primary_terms_csv, index=False)
    complementary_rules_df.to_csv(complementary_rules_csv, index=False)
    complementary_terms_df.to_csv(complementary_terms_csv, index=False)

    summary_payload = {
        'checkpoint_path': str(checkpoint_info['checkpoint_path']),
        'dataset_key': str(checkpoint_info['dataset_key']),
        'fold_idx': int(checkpoint_info['fold_idx']),
        'primary_rule_count': int(len(primary_rules_df)),
        'complementary_rule_count': int(len(complementary_rules_df)),
    }
    summary_json.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding='utf-8')

    lines = []
    lines.append('[Primary / base]')
    for _, row in primary_rules_df.sort_values('rule_idx').iterrows():
        lines.append(f"{row['display_rule_id']} ({row['internal_rule_id']}): {row['if_then_text']}")
    lines.append('')
    lines.append('[Complementary / residual]')
    for _, row in complementary_rules_df.sort_values('rule_idx').iterrows():
        lines.append(f"{row['display_rule_id']} ({row['internal_rule_id']}): {row['if_then_text']}")
    if_then_txt.write_text('\n'.join(lines), encoding='utf-8')

    return {
        'primary_rules_csv': primary_rules_csv,
        'primary_terms_csv': primary_terms_csv,
        'complementary_rules_csv': complementary_rules_csv,
        'complementary_terms_csv': complementary_terms_csv,
        'summary_json': summary_json,
        'if_then_txt': if_then_txt,
    }


In [5]:
resolved_checkpoint_path = resolve_checkpoint_path(DATASET_KEY, FOLD_IDX, CHECKPOINT_PATH)
model, checkpoint_feature_names, gh_params, payload = load_gh_checkpoint(resolved_checkpoint_path, DEVICE)

base_mean_w = None
residual_mean_w = None
snapshot_error = None

if COMPUTE_MEAN_RULE_ACTIVATION:
    try:
        X_snapshot, y_snapshot, snapshot_feature_names = load_snapshot_bundle(DATASET_KEY)
        X_aligned = align_snapshot_to_checkpoint(X_snapshot, checkpoint_feature_names)
        x_tensor = apply_saved_scaler(X_aligned, payload, DEVICE)
        base_mean_w, residual_mean_w = collect_rule_activation_stats(model, x_tensor)
    except Exception as exc:
        snapshot_error = exc

checkpoint_info = {
    'checkpoint_path': resolved_checkpoint_path,
    'dataset_key': DATASET_KEY,
    'fold_idx': int(FOLD_IDX),
}

print('Resolved checkpoint:', resolved_checkpoint_path)
print('n_features:', model.n_features)
print('n_outputs:', model.n_outputs)
print('base_rules (Primary):', model.base_rules)
print('residual_rules (Complementary):', model.residual_rules)
print('mf_per_feature:', model.mf_per_feature)
print('residual_gate_mode:', model.residual_gate_mode)
print('base_mask_threshold:', model.base_mask_threshold)
print('residual_mask_threshold:', model.residual_mask_threshold)
print('feature_names[:10]:', checkpoint_feature_names[:10])

if snapshot_error is None and COMPUTE_MEAN_RULE_ACTIVATION:
    print('Mean rule activation was computed from the local dataset snapshot.')
elif COMPUTE_MEAN_RULE_ACTIVATION:
    print('Mean rule activation was skipped due to:', repr(snapshot_error))


Resolved checkpoint: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/hyper_parameter/cv_weights/Breast_Cancer_Wisconsin__Original___gh_only_5fold/fold_01/gh_anfis.pt
n_features: 80
n_outputs: 1
base_rules (Primary): 6
residual_rules (Complementary): 11
mf_per_feature: 2
residual_gate_mode: complement
base_mask_threshold: 0.5
residual_mask_threshold: 0.5
feature_names[:10]: ['Bare_nuclei', 'Clump_thickness=1', 'Clump_thickness=2', 'Clump_thickness=3', 'Clump_thickness=4', 'Clump_thickness=5', 'Clump_thickness=6', 'Clump_thickness=7', 'Clump_thickness=8', 'Clump_thickness=9']
Mean rule activation was computed from the local dataset snapshot.


In [6]:
primary_rules_df, primary_terms_df = module_rule_tables(
    model,
    checkpoint_feature_names,
    module='base',
    mean_rule_weights=base_mean_w,
    top_terms=TOP_TERMS_PER_RULE,
)

complementary_rules_df, complementary_terms_df = module_rule_tables(
    model,
    checkpoint_feature_names,
    module='residual',
    mean_rule_weights=residual_mean_w,
    top_terms=TOP_TERMS_PER_RULE,
)

print('Primary rules table')
display(primary_rules_df)

print('Complementary rules table')
display(complementary_rules_df)

print('Primary term table')
display(primary_terms_df)

print('Complementary term table')
display(complementary_terms_df)


Primary rules table


,module,module_label,rule_idx,display_rule_id,internal_rule_id,active_feature_count,antecedent_text,consequent_text,if_then_text,predicted_class,class_1_probability,class_probs,rule_logits,mean_rule_activation
0,base,Primary,0,P1,B1,32,Bare_nuclei is low AND Clump_thickness=1 is lo...,"class_1 probability = 0.7298, predicted_class ...",IF Bare_nuclei is low AND Clump_thickness=1 is...,1,7.298138e-01,"{""class_0"": 0.2701861743668942, ""class_1"": 0.7...","{""class_0"": 0.9936782121658325}",0.146237
1,base,Primary,1,P2,B2,32,Bare_nuclei is high AND Clump_thickness=1 is h...,"class_1 probability = 0.0000, predicted_class ...",IF Bare_nuclei is high AND Clump_thickness=1 i...,0,6.094336e-11,"{""class_0"": 0.9999999999390566, ""class_1"": 6.0...","{""class_0"": -23.521076202392578}",0.188669
2,base,Primary,2,P3,B3,32,Bare_nuclei is low AND Clump_thickness=1 is hi...,"class_1 probability = 0.9995, predicted_class ...",IF Bare_nuclei is low AND Clump_thickness=1 is...,1,9.994735e-01,"{""class_0"": 0.0005265149662003754, ""class_1"": ...","{""class_0"": 7.548704147338867}",0.083828
3,base,Primary,3,P4,B4,32,Bare_nuclei is high AND Clump_thickness=1 is l...,"class_1 probability = 0.9965, predicted_class ...",IF Bare_nuclei is high AND Clump_thickness=1 i...,1,9.965445e-01,"{""class_0"": 0.0034554883771720224, ""class_1"": ...","{""class_0"": 5.664330005645752}",0.212069
4,base,Primary,4,P5,B5,32,Bare_nuclei is low AND Clump_thickness=1 is lo...,"class_1 probability = 0.0016, predicted_class ...",IF Bare_nuclei is low AND Clump_thickness=1 is...,0,1.604251e-03,"{""class_0"": 0.9983957488425894, ""class_1"": 0.0...","{""class_0"": -6.433492660522461}",0.182547
5,base,Primary,5,P6,B6,32,Bare_nuclei is high AND Clump_thickness=1 is h...,"class_1 probability = 1.0000, predicted_class ...",IF Bare_nuclei is high AND Clump_thickness=1 i...,1,9.999999e-01,"{""class_0"": 7.891799436166025e-08, ""class_1"": ...","{""class_0"": 16.354856491088867}",0.186648


Complementary rules table


,module,module_label,rule_idx,display_rule_id,internal_rule_id,active_feature_count,antecedent_text,consequent_text,if_then_text,predicted_class,class_1_probability,class_probs,rule_logits,mean_rule_activation
0,residual,Complementary,0,C1,R1,4,Marginal_adhesion=5 is low AND Single_epitheli...,"class_1 probability = 0.6031, predicted_class ...",IF Marginal_adhesion=5 is low AND Single_epith...,1,0.603060,"{""class_0"": 0.39694038088615, ""class_1"": 0.603...","{""class_0"": 0.41822996735572815}",0.091060
1,residual,Complementary,1,C2,R2,4,Marginal_adhesion=5 is low AND Single_epitheli...,"class_1 probability = 0.4706, predicted_class ...",IF Marginal_adhesion=5 is low AND Single_epith...,0,0.470608,"{""class_0"": 0.5293922793077417, ""class_1"": 0.4...","{""class_0"": -0.11770482361316681}",0.090789
2,residual,Complementary,2,C3,R3,4,Marginal_adhesion=5 is high AND Single_epithel...,"class_1 probability = 0.3380, predicted_class ...",IF Marginal_adhesion=5 is high AND Single_epit...,0,0.338004,"{""class_0"": 0.6619959129907682, ""class_1"": 0.3...","{""class_0"": -0.67220139503479}",0.090754
3,residual,Complementary,3,C4,R4,4,Marginal_adhesion=5 is high AND Single_epithel...,"class_1 probability = 0.6298, predicted_class ...",IF Marginal_adhesion=5 is high AND Single_epit...,1,0.629781,"{""class_0"": 0.3702185134422774, ""class_1"": 0.6...","{""class_0"": 0.5312795042991638}",0.091185
4,residual,Complementary,4,C5,R5,4,Marginal_adhesion=5 is high AND Single_epithel...,"class_1 probability = 0.3392, predicted_class ...",IF Marginal_adhesion=5 is high AND Single_epit...,0,0.339158,"{""class_0"": 0.6608419173101987, ""class_1"": 0.3...","{""class_0"": -0.6670483350753784}",0.090754
5,residual,Complementary,5,C6,R6,4,Marginal_adhesion=5 is low AND Single_epitheli...,"class_1 probability = 0.5612, predicted_class ...",IF Marginal_adhesion=5 is low AND Single_epith...,1,0.561209,"{""class_0"": 0.43879113357662847, ""class_1"": 0....","{""class_0"": 0.24606962502002716}",0.090789
6,residual,Complementary,6,C7,R7,4,Marginal_adhesion=5 is low AND Single_epitheli...,"class_1 probability = 0.6049, predicted_class ...",IF Marginal_adhesion=5 is low AND Single_epith...,1,0.604880,"{""class_0"": 0.39512001864219104, ""class_1"": 0....","{""class_0"": 0.42584049701690674}",0.090789
7,residual,Complementary,7,C8,R8,4,Marginal_adhesion=5 is high AND Single_epithel...,"class_1 probability = 0.4774, predicted_class ...",IF Marginal_adhesion=5 is high AND Single_epit...,0,0.477376,"{""class_0"": 0.5226240018767431, ""class_1"": 0.4...","{""class_0"": -0.09055784344673157}",0.090984
8,residual,Complementary,8,C9,R9,4,Marginal_adhesion=5 is low AND Single_epitheli...,"class_1 probability = 0.4597, predicted_class ...",IF Marginal_adhesion=5 is low AND Single_epith...,0,0.459702,"{""class_0"": 0.5402975373850482, ""class_1"": 0.4...","{""class_0"": -0.16154052317142487}",0.090958
9,residual,Complementary,9,C10,R10,4,Marginal_adhesion=5 is high AND Single_epithel...,"class_1 probability = 0.7298, predicted_class ...",IF Marginal_adhesion=5 is high AND Single_epit...,1,0.729837,"{""class_0"": 0.27016305648873273, ""class_1"": 0....","{""class_0"": 0.9937954545021057}",0.091185


Primary term table


,module,module_label,rule_idx,display_rule_id,internal_rule_id,feature,selector_prob,selected_mf_idx,selected_center,term_label,term_polarity_sign,term_text,predicted_class,class_1_probability,consequent_text,mean_rule_activation
0,base,Primary,0,P1,B1,Bare_nuclei,1.0,0,-0.003069,low,-1,Bare_nuclei is low,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
1,base,Primary,0,P1,B1,Clump_thickness=1,1.0,1,-0.731750,low,-1,Clump_thickness=1 is low,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
2,base,Primary,0,P1,B1,Clump_thickness=5,1.0,1,1.912388,high,1,Clump_thickness=5 is high,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
3,base,Primary,0,P1,B1,Clump_thickness=7,1.0,1,0.742817,high,1,Clump_thickness=7 is high,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
4,base,Primary,0,P1,B1,Clump_thickness=8,1.0,1,1.631689,high,1,Clump_thickness=8 is high,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
5,base,Primary,0,P1,B1,Clump_thickness=9,1.0,1,-0.687039,low,-1,Clump_thickness=9 is low,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
6,base,Primary,0,P1,B1,Clump_thickness=10,1.0,1,-0.196535,low,-1,Clump_thickness=10 is low,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
7,base,Primary,0,P1,B1,Uniformity_of_cell_size=1,1.0,0,-1.142469,low,-1,Uniformity_of_cell_size=1 is low,1,7.298138e-01,"class_1 probability = 0.7298, predicted_class ...",0.146237
8,base,Primary,1,P2,B2,Bare_nuclei,1.0,1,2.041466,high,1,Bare_nuclei is high,0,6.094336e-11,"class_1 probability = 0.0000, predicted_class ...",0.188669
9,base,Primary,1,P2,B2,Clump_thickness=1,1.0,0,0.803056,high,1,Clump_thickness=1 is high,0,6.094336e-11,"class_1 probability = 0.0000, predicted_class ...",0.188669


Complementary term table


,module,module_label,rule_idx,display_rule_id,internal_rule_id,feature,selector_prob,selected_mf_idx,selected_center,term_label,term_polarity_sign,term_text,predicted_class,class_1_probability,consequent_text,mean_rule_activation
0,residual,Complementary,0,C1,R1,Marginal_adhesion=5,1.0,0,-1.154999,low,-1,Marginal_adhesion=5 is low,1,0.603060,"class_1 probability = 0.6031, predicted_class ...",0.091060
1,residual,Complementary,0,C1,R1,Single_epithelial_cell_size=1,1.0,1,1.154127,high,1,Single_epithelial_cell_size=1 is high,1,0.603060,"class_1 probability = 0.6031, predicted_class ...",0.091060
2,residual,Complementary,0,C1,R1,Normal_nucleoli=7,1.0,1,1.159698,high,1,Normal_nucleoli=7 is high,1,0.603060,"class_1 probability = 0.6031, predicted_class ...",0.091060
3,residual,Complementary,0,C1,R1,Mitoses=4,1.0,1,1.156280,high,1,Mitoses=4 is high,1,0.603060,"class_1 probability = 0.6031, predicted_class ...",0.091060
4,residual,Complementary,1,C2,R2,Marginal_adhesion=5,1.0,0,-1.154999,low,-1,Marginal_adhesion=5 is low,0,0.470608,"class_1 probability = 0.4706, predicted_class ...",0.090789
5,residual,Complementary,1,C2,R2,Single_epithelial_cell_size=1,1.0,1,1.154127,high,1,Single_epithelial_cell_size=1 is high,0,0.470608,"class_1 probability = 0.4706, predicted_class ...",0.090789
6,residual,Complementary,1,C2,R2,Normal_nucleoli=7,1.0,1,1.159698,high,1,Normal_nucleoli=7 is high,0,0.470608,"class_1 probability = 0.4706, predicted_class ...",0.090789
7,residual,Complementary,1,C2,R2,Mitoses=4,1.0,0,-1.156689,low,-1,Mitoses=4 is low,0,0.470608,"class_1 probability = 0.4706, predicted_class ...",0.090789
8,residual,Complementary,2,C3,R3,Marginal_adhesion=5,1.0,1,1.156202,high,1,Marginal_adhesion=5 is high,0,0.338004,"class_1 probability = 0.3380, predicted_class ...",0.090754
9,residual,Complementary,2,C3,R3,Single_epithelial_cell_size=1,1.0,0,-1.157353,low,-1,Single_epithelial_cell_size=1 is low,0,0.338004,"class_1 probability = 0.3380, predicted_class ...",0.090754


In [7]:
print_if_then_rules(primary_rules_df, 'Primary / base')
print()
print_if_then_rules(complementary_rules_df, 'Complementary / residual')


[Primary / base]
- P1 (B1): IF Bare_nuclei is low AND Clump_thickness=1 is low AND Clump_thickness=5 is high AND Clump_thickness=7 is high AND Clump_thickness=8 is high AND Clump_thickness=9 is low AND Clump_thickness=10 is low AND Uniformity_of_cell_size=1 is low THEN class_1 probability = 0.7298, predicted_class = class_1, logit = 0.9937
- P2 (B2): IF Bare_nuclei is high AND Clump_thickness=1 is high AND Clump_thickness=5 is low AND Clump_thickness=7 is low AND Clump_thickness=8 is high AND Clump_thickness=9 is low AND Clump_thickness=10 is low AND Uniformity_of_cell_size=1 is low THEN class_1 probability = 0.0000, predicted_class = class_0, logit = -23.5211
- P3 (B3): IF Bare_nuclei is low AND Clump_thickness=1 is high AND Clump_thickness=5 is low AND Clump_thickness=7 is high AND Clump_thickness=8 is low AND Clump_thickness=9 is high AND Clump_thickness=10 is high AND Uniformity_of_cell_size=1 is high THEN class_1 probability = 0.9995, predicted_class = class_1, logit = 7.5487
- P4

In [8]:
stem = f'{DATASET_KEY}_fold_{int(FOLD_IDX):02d}'
exported_paths = export_rule_outputs(
    export_dir=EXPORT_DIR,
    stem=stem,
    primary_rules_df=primary_rules_df,
    primary_terms_df=primary_terms_df,
    complementary_rules_df=complementary_rules_df,
    complementary_terms_df=complementary_terms_df,
    checkpoint_info=checkpoint_info,
)

print('Exported files:')
for key, path in exported_paths.items():
    print(f'  {key}: {path.resolve()}')


Exported files:
  primary_rules_csv: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/grs_rule_if_then/Breast_Cancer_Wisconsin__Original___gh_only_5fold_fold_01_primary_rules.csv
  primary_terms_csv: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/grs_rule_if_then/Breast_Cancer_Wisconsin__Original___gh_only_5fold_fold_01_primary_terms.csv
  complementary_rules_csv: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/grs_rule_if_then/Breast_Cancer_Wisconsin__Original___gh_only_5fold_fold_01_complementary_rules.csv
  complementary_terms_csv: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/grs_rule_if_then/Breast_Cancer_Wisconsin__Original___gh_only_5fold_fold_01_complementary_terms.csv
  summary_json: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/grs_rule_if_then/Breast_Cancer_Wisconsin__Original___gh_only_5fold_fold_01_summary.json
  if_then_txt: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/grs_rule_if_then/Breast_Cancer_Wiscon